In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import os

# ── Project Paths ──────────────────────────────────────────────────
DATA_DIR         = r"C:\Data Science\DS Projects\Health vs Wealth analysis"
CHART_DIR_WHO    = r"C:\Data Science\DS Projects\Health vs Wealth analysis\Charts\WHO"
CHART_DIR_MERGED = r"C:\Data Science\DS Projects\Health vs Wealth analysis\Charts\Merged"

WHO_FILE  = os.path.join(DATA_DIR, "Life_Expectancy_Data_1_CSV.csv")

GDP_FILE  = os.path.join(DATA_DIR, "GDP per capita main CSV.csv")
POP_FILE  = os.path.join(DATA_DIR, "Population main CSV.csv")

# ── Create folder structure ────────────────────────────────────────
os.makedirs(CHART_DIR_WHO,    exist_ok=True)
os.makedirs(CHART_DIR_MERGED, exist_ok=True)

# ── Load Data ──────────────────────────────────────────────────────
who    = pd.read_csv(WHO_FILE)

gdp_wb = pd.read_csv(GDP_FILE, skiprows=4)
pop_wb = pd.read_csv(POP_FILE, skiprows=4)

# ── Clean column names ─────────────────────────────────────────────
who.columns = who.columns.str.strip()


print("WHO dataset:        ", who.shape)

print("GDP WB shape:       ", gdp_wb.shape)
print("Population shape:   ", pop_wb.shape)
print("Paths confirmed:    ", DATA_DIR)

WHO dataset:         (2938, 22)
GDP WB shape:        (266, 70)
Population shape:    (266, 70)
Paths confirmed:     C:\Data Science\DS Projects\Health vs Wealth analysis


## Hypothesis 1 — Physical Health & Wealth
**Expected relationship:** Monotonic positive correlation

Life expectancy and physical health outcomes will correlate positively with GDP per capita. Wealthier nations will show better outcomes across mortality, disease, and longevity metrics.

## Hypothesis 2 — Mental Health & Wealth
**Expected relationship:** U-shaped curve

Mental health outcomes will follow a U-shaped curve relative to GDP per capita. At low wealth levels, survival needs dominate but provide inherent meaning and purpose. Beyond a threshold of material comfort, meaning and purpose become harder to find and mental health outcomes deteriorate.

**Proposed mechanism:** Survival as a source of meaning — Frankl / logotherapy framework. Abundance removes existential drivers without replacing them.

## Caveats
The WHO dataset may lack direct mental health metrics. Proxy variables such as alcohol consumption and BMI may be used where necessary, with appropriate caution in interpretation. A supplementary dataset may be required for the full mental health analysis.

In [2]:
# ── WHO Data Quality Check ─────────────────────────────────────────
print("Shape:", who.shape)
print("\nData types:")
print(who.dtypes)
print("\nMissing values per column:")
print(who.isnull().sum())
print("\nSummary statistics:")
print(who.describe())

Shape: (2938, 22)

Data types:
Country                             object
Year                                 int64
Status                              object
Life expectancy                    float64
Adult Mortality                    float64
infant deaths                        int64
Alcohol                            float64
percentage expenditure             float64
Hepatitis B                        float64
Measles                              int64
BMI                                float64
under-five deaths                    int64
Polio                              float64
Total expenditure                  float64
Diphtheria                         float64
HIV/AIDS                           float64
GDP                                float64
Population                         float64
thinness  1-19 years               float64
thinness 5-9 years                 float64
Income composition of resources    float64
Schooling                          float64
dtype: object

Missing 

In [3]:
# ── Investigate missing values by year ────────────────────────────
print("Missing GDP values by year:")
print(who[who['GDP'].isnull()]['Year'].value_counts().sort_index())

print("\nMissing Population values by year:")
print(who[who['Population'].isnull()]['Year'].value_counts().sort_index())

# ── Find the suspicious BMI and Population entries ─────────────────
print("\nRows where BMI < 10:")
print(who[who['BMI'] < 10][['Country', 'Year', 'BMI']])

print("\nRows where Population < 1000:")
print(who[who['Population'] < 1000][['Country', 'Year', 'Population']])

Missing GDP values by year:
Year
2000    29
2001    28
2002    28
2003    28
2004    27
2005    27
2006    27
2007    27
2008    27
2009    27
2010    27
2011    27
2012    29
2013    33
2014    28
2015    29
Name: count, dtype: int64

Missing Population values by year:
Year
2000    40
2001    40
2002    40
2003    40
2004    40
2005    40
2006    40
2007    40
2008    40
2009    40
2010    40
2011    40
2012    41
2013    49
2014    41
2015    41
Name: count, dtype: int64

Rows where BMI < 10:
                  Country  Year  BMI
25                Albania  2006  5.8
40                Algeria  2007  5.8
53                 Angola  2010  2.4
75    Antigua and Barbuda  2004  4.8
76    Antigua and Barbuda  2003  4.1
...                   ...   ...  ...
2891                Yemen  2014  4.4
2911               Zambia  2010  2.7
2912               Zambia  2009  2.2
2924             Zimbabwe  2013  3.8
2925             Zimbabwe  2012  3.3

[283 rows x 3 columns]

Rows where Population < 1000:
 

In [4]:
# ── Check population units theory ─────────────────────────────────
# Find countries we know the real population of
known = ['United States of America', 'China', 'India', 'Brazil', 'Germany']
print(who[who['Country'].isin(known)][['Country', 'Year', 'Population']]
      .sort_values(['Country', 'Year'])
      .to_string())

                       Country  Year    Population
367                     Brazil  2000  1.752876e+08
366                     Brazil  2001  1.777567e+06
365                     Brazil  2002  1.815121e+06
364                     Brazil  2003  1.824821e+08
363                     Brazil  2004  1.847385e+08
362                     Brazil  2005  1.869174e+08
361                     Brazil  2006  1.891241e+07
360                     Brazil  2007  1.912664e+07
359                     Brazil  2008  1.929793e+07
358                     Brazil  2009  1.948960e+08
357                     Brazil  2010  1.967963e+08
356                     Brazil  2011  1.986867e+08
355                     Brazil  2012  2.569830e+05
354                     Brazil  2013  2.248632e+06
353                     Brazil  2014  2.421313e+07
352                     Brazil  2015  2.596218e+06
575                      China  2000  1.262645e+06
574                      China  2001  1.271850e+05
573                      China 

In [5]:
# ── Check GDP values for known countries ──────────────────────────
known = ['United States of America', 'China', 'India', 'Brazil', 'Germany']
print(who[who['Country'].isin(known)][['Country', 'Year', 'GDP']]
      .sort_values(['Country', 'Year'])
      .to_string())

                       Country  Year           GDP
367                     Brazil  2000   3739.119360
366                     Brazil  2001   3146.951770
365                     Brazil  2002   2819.649531
364                     Brazil  2003    359.587582
363                     Brazil  2004   3623.476670
362                     Brazil  2005    477.182746
361                     Brazil  2006    586.145975
360                     Brazil  2007   7313.557962
359                     Brazil  2008   8787.613750
358                     Brazil  2009   8553.384700
357                     Brazil  2010  11224.154800
356                     Brazil  2011  13167.472890
355                     Brazil  2012  12291.466850
354                     Brazil  2013  12216.944600
353                     Brazil  2014   1226.617310
352                     Brazil  2015   8757.262200
575                      China  2000    959.372181
574                      China  2001    153.182400
573                      China 

In [6]:
# ── Load World Bank Data ───────────────────────────────────────────
gdp_wb = pd.read_csv(r"C:\Data Science\DS Projects\Health vs Wealth analysis\GDP per capita main CSV.csv", skiprows=4)
pop_wb = pd.read_csv(r"C:\Data Science\DS Projects\Health vs Wealth analysis\Population main CSV.csv", skiprows=4)

print("GDP per capita shape:", gdp_wb.shape)
print("\nGDP columns:")
print(gdp_wb.columns.tolist())

print("\nPopulation shape:", pop_wb.shape)
print("\nFirst 3 rows of GDP:")
print(gdp_wb.head(3).to_string())

GDP per capita shape: (266, 70)

GDP columns:
['Country Name', 'Country Code', 'Indicator Name', 'Indicator Code', '1960', '1961', '1962', '1963', '1964', '1965', '1966', '1967', '1968', '1969', '1970', '1971', '1972', '1973', '1974', '1975', '1976', '1977', '1978', '1979', '1980', '1981', '1982', '1983', '1984', '1985', '1986', '1987', '1988', '1989', '1990', '1991', '1992', '1993', '1994', '1995', '1996', '1997', '1998', '1999', '2000', '2001', '2002', '2003', '2004', '2005', '2006', '2007', '2008', '2009', '2010', '2011', '2012', '2013', '2014', '2015', '2016', '2017', '2018', '2019', '2020', '2021', '2022', '2023', '2024', '2025']

Population shape: (266, 70)

First 3 rows of GDP:
                  Country Name Country Code                Indicator Name  Indicator Code        1960        1961        1962        1963        1964        1965        1966        1967        1968        1969        1970        1971        1972        1973        1974        1975        1976        1977 

In [7]:
import pandas as pd
import matplotlib.pyplot as plt
import os

# ── Project Paths ──────────────────────────────────────────────────
DATA_DIR         = r"C:\Data Science\DS Projects\Health vs Wealth analysis"
CHART_DIR_WHO    = r"C:\Data Science\DS Projects\Health vs Wealth analysis\Charts\WHO"
CHART_DIR_MERGED = r"C:\Data Science\DS Projects\Health vs Wealth analysis\Charts\Merged"

WHO_FILE  = os.path.join(DATA_DIR, "Life_Expectancy_Data_1_CSV.csv")
GDP_FILE  = os.path.join(DATA_DIR, "GDP per capita main CSV.csv")
POP_FILE  = os.path.join(DATA_DIR, "Population main CSV.csv")

# ── Create folder structure ────────────────────────────────────────
os.makedirs(CHART_DIR_WHO,    exist_ok=True)
os.makedirs(CHART_DIR_MERGED, exist_ok=True)

print("Paths set up successfully")
print("Data directory:", DATA_DIR)

Paths set up successfully
Data directory: C:\Data Science\DS Projects\Health vs Wealth analysis


In [8]:
# ── Load Datasets ──────────────────────────────────────────────────
who    = pd.read_csv(WHO_FILE)
gdp_wb = pd.read_csv(GDP_FILE, skiprows=4)
pop_wb = pd.read_csv(POP_FILE, skiprows=4)

# ── Clean WHO column names ─────────────────────────────────────────
who.columns = who.columns.str.strip()

# ── Quick confirmation ─────────────────────────────────────────────
print("WHO shape:        ", who.shape)
print("GDP WB shape:     ", gdp_wb.shape)
print("Population shape: ", pop_wb.shape)

WHO shape:         (2938, 22)
GDP WB shape:      (266, 70)
Population shape:  (266, 70)


In [9]:
import os

for f in os.listdir(r"C:\Data Science\DS Projects\Health vs Wealth analysis"):
    print(f)

.ipynb_checkpoints
02_WHO_Analysis.ipynb
Charts
GDP per capita main CSV.csv
Life_Expectancy_Data_1_CSV.csv
Original datasets
Population main CSV.csv


In [10]:
# ── Inspect World Bank GDP file ────────────────────────────────────
print("GDP columns (first 10):")
print(gdp_wb.columns[:10].tolist())

print("\nGDP columns (last 10):")
print(gdp_wb.columns[-10:].tolist())

print("\nFirst 3 rows, first 6 columns:")
print(gdp_wb.iloc[:3, :6].to_string())

GDP columns (first 10):
['Country Name', 'Country Code', 'Indicator Name', 'Indicator Code', '1960', '1961', '1962', '1963', '1964', '1965']

GDP columns (last 10):
['2016', '2017', '2018', '2019', '2020', '2021', '2022', '2023', '2024', '2025']

First 3 rows, first 6 columns:
                  Country Name Country Code                Indicator Name  Indicator Code        1960        1961
0                        Aruba          ABW  GDP per capita (current US$)  NY.GDP.PCAP.CD         NaN         NaN
1  Africa Eastern and Southern          AFE  GDP per capita (current US$)  NY.GDP.PCAP.CD  186.089204  186.909053
2                  Afghanistan          AFG  GDP per capita (current US$)  NY.GDP.PCAP.CD         NaN         NaN


In [11]:
# ── Reshape GDP from wide to long format ───────────────────────────
gdp_long = gdp_wb.melt(
    id_vars    = ['Country Name', 'Country Code'],
    value_vars = [str(y) for y in range(2000, 2016)],
    var_name   = 'Year',
    value_name = 'GDP_per_capita'
)

# ── Convert Year from string to integer ────────────────────────────
gdp_long['Year'] = gdp_long['Year'].astype(int)

print("GDP long format shape:", gdp_long.shape)
print("\nFirst 5 rows:")
print(gdp_long.head())

GDP long format shape: (4256, 4)

First 5 rows:
                  Country Name Country Code  Year  GDP_per_capita
0                        Aruba          ABW  2000    20681.023027
1  Africa Eastern and Southern          AFE  2000      706.727261
2                  Afghanistan          AFG  2000      174.930991
3   Africa Western and Central          AFW  2000      518.969226
4                       Angola          AGO  2000      563.733796


In [12]:
# ── Reshape Population from wide to long format ────────────────────
pop_long = pop_wb.melt(
    id_vars    = ['Country Name', 'Country Code'],
    value_vars = [str(y) for y in range(2000, 2016)],
    var_name   = 'Year',
    value_name = 'Population'
)

# ── Convert Year from string to integer ────────────────────────────
pop_long['Year'] = pop_long['Year'].astype(int)

print("Population long format shape:", pop_long.shape)
print("\nFirst 5 rows:")
print(pop_long.head())

Population long format shape: (4256, 4)

First 5 rows:
                  Country Name Country Code  Year   Population
0                        Aruba          ABW  2000      90588.0
1  Africa Eastern and Southern          AFE  2000  406156661.0
2                  Afghanistan          AFG  2000   20130327.0
3   Africa Western and Central          AFW  2000  274968446.0
4                       Angola          AGO  2000   16194869.0


In [13]:
# ── Spot check population values ──────────────────────────────────
check_countries = ['United States', 'China', 'India', 'Brazil', 'Germany']

print("Population spot check:")
print(pop_long[
    (pop_long['Country Name'].isin(check_countries)) &
    (pop_long['Year'] == 2010)
][['Country Name', 'Year', 'Population']].to_string())

Population spot check:
       Country Name  Year    Population
2689         Brazil  2010  1.937019e+08
2700          China  2010  1.337705e+09
2715        Germany  2010  8.177693e+07
2769          India  2010  1.243482e+09
2911  United States  2010  3.093782e+08


In [14]:
# ── Check for country name mismatches ─────────────────────────────
who_countries = set(who['Country'].unique())
wb_countries  = set(gdp_long['Country Name'].unique())

in_both    = who_countries & wb_countries
in_who_only = who_countries - wb_countries
in_wb_only  = wb_countries - who_countries

print(f"Countries in both:     {len(in_both)}")
print(f"Countries in WHO only: {len(in_who_only)}")
print(f"\nSample of WHO-only countries:")
print(sorted(in_who_only)[:20])
print(f"\nSample of WB-only countries:")
print(sorted(in_wb_only)[:20])

Countries in both:     164
Countries in WHO only: 29

Sample of WHO-only countries:
['Bahamas', 'Bolivia (Plurinational State of)', 'Congo', 'Cook Islands', "CÃ´te d'Ivoire", "Democratic People's Republic of Korea", 'Democratic Republic of the Congo', 'Egypt', 'Gambia', 'Iran (Islamic Republic of)', 'Kyrgyzstan', "Lao People's Democratic Republic", 'Micronesia (Federated States of)', 'Niue', 'Republic of Korea', 'Republic of Moldova', 'Saint Kitts and Nevis', 'Saint Lucia', 'Saint Vincent and the Grenadines', 'Slovakia']

Sample of WB-only countries:
['Africa Eastern and Southern', 'Africa Western and Central', 'American Samoa', 'Andorra', 'Arab World', 'Aruba', 'Bahamas, The', 'Bermuda', 'Bolivia', 'British Virgin Islands', 'Caribbean small states', 'Cayman Islands', 'Central Europe and the Baltics', 'Channel Islands', 'Congo, Dem. Rep.', 'Congo, Rep.', "Cote d'Ivoire", 'Curacao', 'Early-demographic dividend', 'East Asia & Pacific']


In [15]:
print(who.columns.tolist())

['Country', 'Year', 'Status', 'Life expectancy', 'Adult Mortality', 'infant deaths', 'Alcohol', 'percentage expenditure', 'Hepatitis B', 'Measles', 'BMI', 'under-five deaths', 'Polio', 'Total expenditure', 'Diphtheria', 'HIV/AIDS', 'GDP', 'Population', 'thinness  1-19 years', 'thinness 5-9 years', 'Income composition of resources', 'Schooling']


In [16]:
print("WHO columns:")
print(who.columns.tolist())

print("\nGDP Long columns:")
print(gdp_long.columns.tolist())

print("\nPop Long columns:")
print(pop_long.columns.tolist())

WHO columns:
['Country', 'Year', 'Status', 'Life expectancy', 'Adult Mortality', 'infant deaths', 'Alcohol', 'percentage expenditure', 'Hepatitis B', 'Measles', 'BMI', 'under-five deaths', 'Polio', 'Total expenditure', 'Diphtheria', 'HIV/AIDS', 'GDP', 'Population', 'thinness  1-19 years', 'thinness 5-9 years', 'Income composition of resources', 'Schooling']

GDP Long columns:
['Country Name', 'Country Code', 'Year', 'GDP_per_capita']

Pop Long columns:
['Country Name', 'Country Code', 'Year', 'Population']


In [17]:
# ── Full list of WHO countries with no World Bank match ───────────
print("All WHO-only countries:")
print(sorted(in_who_only))

All WHO-only countries:
['Bahamas', 'Bolivia (Plurinational State of)', 'Congo', 'Cook Islands', "CÃ´te d'Ivoire", "Democratic People's Republic of Korea", 'Democratic Republic of the Congo', 'Egypt', 'Gambia', 'Iran (Islamic Republic of)', 'Kyrgyzstan', "Lao People's Democratic Republic", 'Micronesia (Federated States of)', 'Niue', 'Republic of Korea', 'Republic of Moldova', 'Saint Kitts and Nevis', 'Saint Lucia', 'Saint Vincent and the Grenadines', 'Slovakia', 'Somalia', 'Swaziland', 'The former Yugoslav republic of Macedonia', 'Turkey', 'United Kingdom of Great Britain and Northern Ireland', 'United Republic of Tanzania', 'United States of America', 'Venezuela (Bolivarian Republic of)', 'Yemen']


In [18]:
# ── Rename WHO countries to match World Bank naming ───────────────
who['Country'] = who['Country'].replace({
    'Bahamas'                                        : 'Bahamas, The',
    'Bolivia (Plurinational State of)'               : 'Bolivia',
    'Congo'                                          : 'Congo, Rep.',
    'Democratic Republic of the Congo'               : 'Congo, Dem. Rep.',
    'CÃ´te d\'Ivoire'                                : "Cote d'Ivoire",
    'Egypt'                                          : 'Egypt, Arab Rep.',
    'Gambia'                                         : 'Gambia, The',
    'Iran (Islamic Republic of)'                     : 'Iran, Islamic Rep.',
    'Kyrgyzstan'                                     : 'Kyrgyz Republic',
    "Lao People's Democratic Republic"               : 'Lao PDR',
    'Micronesia (Federated States of)'               : 'Micronesia, Fed. Sts.',
    'Republic of Korea'                              : 'Korea, Rep.',
    'Republic of Moldova'                            : 'Moldova',
    "Democratic People's Republic of Korea"          : "Korea, Dem. People's Rep.",
    'Slovakia'                                       : 'Slovak Republic',
    'Swaziland'                                      : 'Eswatini',
    'The former Yugoslav republic of Macedonia'      : 'North Macedonia',
    'Turkey'                                         : 'Turkiye',
    'United Kingdom of Great Britain and Northern Ireland' : 'United Kingdom',
    'United Republic of Tanzania'                    : 'Tanzania',
    'United States of America'                       : 'United States',
    'Venezuela (Bolivarian Republic of)'             : 'Venezuela, RB',
    'Yemen'                                          : 'Yemen, Rep.'
})

# ── Recheck mismatches after renaming ─────────────────────────────
who_countries = set(who['Country'].unique())
wb_countries  = set(gdp_long['Country Name'].unique())
in_who_only   = who_countries - wb_countries

print(f"Remaining WHO-only countries after renaming: {len(in_who_only)}")
print(sorted(in_who_only))

Remaining WHO-only countries after renaming: 6
['Cook Islands', 'Niue', 'Saint Kitts and Nevis', 'Saint Lucia', 'Saint Vincent and the Grenadines', 'Somalia']


In [19]:
# ── Step 1: Build country code lookup table from World Bank ────────
lookup = gdp_long[['Country Name', 'Country Code']].drop_duplicates()

# ── Merge country codes into WHO ───────────────────────────────────
who = pd.merge(who, lookup, left_on='Country', right_on='Country Name', how='left')

print("WHO shape after adding country codes:", who.shape)
print("\nMissing country codes:")
print(who[who['Country Code'].isnull()]['Country'].unique())

WHO shape after adding country codes: (2938, 24)

Missing country codes:
['Cook Islands' 'Niue' 'Saint Kitts and Nevis' 'Saint Lucia'
 'Saint Vincent and the Grenadines' 'Somalia']


In [20]:
# ── Step 2: Merge GDP per capita into WHO ─────────────────────────
who = pd.merge(who, 
               gdp_long[['Country Code', 'Year', 'GDP_per_capita']], 
               on=['Country Code', 'Year'], 
               how='left')

# ── Merge Population into WHO ──────────────────────────────────────
who = pd.merge(who, 
               pop_long[['Country Code', 'Year', 'Population']], 
               on=['Country Code', 'Year'], 
               how='left')

print("WHO shape after merging GDP and Population:", who.shape)
print("\nMissing GDP values:", who['GDP_per_capita'].isnull().sum())
print("Missing Population values:", who['Population'].isnull().sum())

WHO shape after merging GDP and Population: (2938, 26)

Missing GDP values: 79


KeyError: 'Population'

In [ ]:
print(who.columns.tolist())

In [ ]:
# ── Drop unreliable original columns ──────────────────────────────
who = who.drop(columns=['GDP', 'Population_x', 'Country Name'])

# ── Rename clean World Bank population column ──────────────────────
who = who.rename(columns={'Population_y' : 'Population'})

# ── Confirm ────────────────────────────────────────────────────────
print("Final columns:")
print(who.columns.tolist())
print("\nFinal shape:", who.shape)

In [ ]:
print("Missing values per column:")
print(who.isnull().sum())

In [ ]:
print("who" in dir())
print("gdp_long" in dir())
print("pop_long" in dir())

In [ ]:
print("who shape before merge:", who.shape)
print("who columns:", who.columns.tolist())
print("\ngdp_long columns:", gdp_long.columns.tolist())

In [ ]:
# ── Drop unreliable original columns ──────────────────────────────
who = who.drop(columns=['GDP', 'Population_x', 'Country Name'])

# ── Rename clean World Bank population column ──────────────────────
who = who.rename(columns={'Population_y' : 'Population'})

# ── Confirm ────────────────────────────────────────────────────────
print("Final columns:")
print(who.columns.tolist())
print("\nFinal shape:", who.shape)

In [ ]:
import matplotlib.pyplot as plt

# ── Scatter plot: GDP per capita vs Life Expectancy ────────────────
fig, ax = plt.subplots(figsize=(10, 6))

ax.scatter(
    who['GDP_per_capita'],
    who['Life expectancy'],
    alpha=0.3,
    s=10,
    color='steelblue'
)

ax.set_xlabel('GDP per Capita (USD)')
ax.set_ylabel('Life Expectancy (Years)')
ax.set_title('Life Expectancy vs GDP per Capita\nWHO Dataset 2000–2015')

plt.tight_layout()
plt.savefig(os.path.join(CHART_DIR_WHO, 'life_exp_vs_gdp.png'), dpi=150, bbox_inches='tight')
plt.show()

print("Chart saved.")

In [ ]:
# ── Who are the highest GDP per capita countries? ─────────────────
print(who[who['GDP_per_capita'] > 50000][['Country', 'Year', 'GDP_per_capita', 'Life expectancy']]
      .sort_values('GDP_per_capita', ascending=False)
      .head(20)
      .to_string())

In [ ]:
print(sorted(who['Country'].unique()))

In [ ]:
# ── Region mapping ─────────────────────────────────────────────────
region_map = {
    # Sub-Saharan Africa
    'Angola':'Sub-Saharan Africa', 'Benin':'Sub-Saharan Africa',
    'Botswana':'Sub-Saharan Africa', 'Burkina Faso':'Sub-Saharan Africa',
    'Burundi':'Sub-Saharan Africa', 'Cabo Verde':'Sub-Saharan Africa',
    'Cameroon':'Sub-Saharan Africa', 'Central African Republic':'Sub-Saharan Africa',
    'Chad':'Sub-Saharan Africa', 'Comoros':'Sub-Saharan Africa',
    'Congo, Dem. Rep.':'Sub-Saharan Africa', 'Congo, Rep.':'Sub-Saharan Africa',
    "Cote d'Ivoire":'Sub-Saharan Africa', 'Djibouti':'Sub-Saharan Africa',
    'Equatorial Guinea':'Sub-Saharan Africa', 'Eritrea':'Sub-Saharan Africa',
    'Eswatini':'Sub-Saharan Africa', 'Ethiopia':'Sub-Saharan Africa',
    'Gabon':'Sub-Saharan Africa', 'Gambia, The':'Sub-Saharan Africa',
    'Ghana':'Sub-Saharan Africa', 'Guinea':'Sub-Saharan Africa',
    'Guinea-Bissau':'Sub-Saharan Africa', 'Kenya':'Sub-Saharan Africa',
    'Lesotho':'Sub-Saharan Africa', 'Liberia':'Sub-Saharan Africa',
    'Madagascar':'Sub-Saharan Africa', 'Malawi':'Sub-Saharan Africa',
    'Mali':'Sub-Saharan Africa', 'Mauritania':'Sub-Saharan Africa',
    'Mauritius':'Sub-Saharan Africa', 'Mozambique':'Sub-Saharan Africa',
    'Namibia':'Sub-Saharan Africa', 'Niger':'Sub-Saharan Africa',
    'Nigeria':'Sub-Saharan Africa', 'Rwanda':'Sub-Saharan Africa',
    'Sao Tome and Principe':'Sub-Saharan Africa', 'Senegal':'Sub-Saharan Africa',
    'Seychelles':'Sub-Saharan Africa', 'Sierra Leone':'Sub-Saharan Africa',
    'Somalia':'Sub-Saharan Africa', 'South Africa':'Sub-Saharan Africa',
    'South Sudan':'Sub-Saharan Africa', 'Sudan':'Sub-Saharan Africa',
    'Tanzania':'Sub-Saharan Africa', 'Togo':'Sub-Saharan Africa',
    'Uganda':'Sub-Saharan Africa', 'Zambia':'Sub-Saharan Africa',
    'Zimbabwe':'Sub-Saharan Africa',

    # North Africa & Middle East
    'Algeria':'North Africa & Middle East', 'Bahrain':'North Africa & Middle East',
    'Egypt, Arab Rep.':'North Africa & Middle East', 'Iran, Islamic Rep.':'North Africa & Middle East',
    'Iraq':'North Africa & Middle East', 'Jordan':'North Africa & Middle East',
    'Kuwait':'North Africa & Middle East', 'Lebanon':'North Africa & Middle East',
    'Libya':'North Africa & Middle East', 'Morocco':'North Africa & Middle East',
    'Oman':'North Africa & Middle East', 'Qatar':'North Africa & Middle East',
    'Saudi Arabia':'North Africa & Middle East', 'Syrian Arab Republic':'North Africa & Middle East',
    'Tunisia':'North Africa & Middle East', 'United Arab Emirates':'North Africa & Middle East',
    'Yemen, Rep.':'North Africa & Middle East',

    # Europe & Central Asia
    'Albania':'Europe & Central Asia', 'Armenia':'Europe & Central Asia',
    'Austria':'Europe & Central Asia', 'Azerbaijan':'Europe & Central Asia',
    'Belarus':'Europe & Central Asia', 'Belgium':'Europe & Central Asia',
    'Bosnia and Herzegovina':'Europe & Central Asia', 'Bulgaria':'Europe & Central Asia',
    'Croatia':'Europe & Central Asia', 'Cyprus':'Europe & Central Asia',
    'Czechia':'Europe & Central Asia', 'Denmark':'Europe & Central Asia',
    'Estonia':'Europe & Central Asia', 'Finland':'Europe & Central Asia',
    'France':'Europe & Central Asia', 'Georgia':'Europe & Central Asia',
    'Germany':'Europe & Central Asia', 'Greece':'Europe & Central Asia',
    'Hungary':'Europe & Central Asia', 'Iceland':'Europe & Central Asia',
    'Ireland':'Europe & Central Asia', 'Israel':'Europe & Central Asia',
    'Italy':'Europe & Central Asia', 'Kazakhstan':'Europe & Central Asia',
    'Kyrgyz Republic':'Europe & Central Asia', 'Latvia':'Europe & Central Asia',
    'Lithuania':'Europe & Central Asia', 'Luxembourg':'Europe & Central Asia',
    'Malta':'Europe & Central Asia', 'Moldova':'Europe & Central Asia',
    'Monaco':'Europe & Central Asia', 'Montenegro':'Europe & Central Asia',
    'Netherlands':'Europe & Central Asia', 'North Macedonia':'Europe & Central Asia',
    'Norway':'Europe & Central Asia', 'Poland':'Europe & Central Asia',
    'Portugal':'Europe & Central Asia', 'Romania':'Europe & Central Asia',
    'Russian Federation':'Europe & Central Asia', 'San Marino':'Europe & Central Asia',
    'Serbia':'Europe & Central Asia', 'Slovak Republic':'Europe & Central Asia',
    'Slovenia':'Europe & Central Asia', 'Spain':'Europe & Central Asia',
    'Sweden':'Europe & Central Asia', 'Switzerland':'Europe & Central Asia',
    'Tajikistan':'Europe & Central Asia', 'Turkiye':'Europe & Central Asia',
    'Turkmenistan':'Europe & Central Asia', 'Ukraine':'Europe & Central Asia',
    'United Kingdom':'Europe & Central Asia', 'Uzbekistan':'Europe & Central Asia',

    # South & East Asia
    'Afghanistan':'South & East Asia', 'Bangladesh':'South & East Asia',
    'Bhutan':'South & East Asia', 'Brunei Darussalam':'South & East Asia',
    'Cambodia':'South & East Asia', 'China':'South & East Asia',
    'India':'South & East Asia', 'Indonesia':'South & East Asia',
    'Japan':'South & East Asia', "Korea, Dem. People's Rep.":'South & East Asia',
    'Korea, Rep.':'South & East Asia', 'Lao PDR':'South & East Asia',
    'Malaysia':'South & East Asia', 'Maldives':'South & East Asia',
    'Mongolia':'South & East Asia', 'Myanmar':'South & East Asia',
    'Nepal':'South & East Asia', 'Pakistan':'South & East Asia',
    'Philippines':'South & East Asia', 'Singapore':'South & East Asia',
    'Sri Lanka':'South & East Asia', 'Thailand':'South & East Asia',
    'Timor-Leste':'South & East Asia', 'Viet Nam':'South & East Asia',

    # Americas
    'Antigua and Barbuda':'Americas', 'Argentina':'Americas',
    'Bahamas, The':'Americas', 'Barbados':'Americas',
    'Belize':'Americas', 'Bolivia':'Americas',
    'Brazil':'Americas', 'Canada':'Americas',
    'Chile':'Americas', 'Colombia':'Americas',
    'Costa Rica':'Americas', 'Cuba':'Americas',
    'Dominica':'Americas', 'Dominican Republic':'Americas',
    'Ecuador':'Americas', 'El Salvador':'Americas',
    'Grenada':'Americas', 'Guatemala':'Americas',
    'Guyana':'Americas', 'Haiti':'Americas',
    'Honduras':'Americas', 'Jamaica':'Americas',
    'Mexico':'Americas', 'Nicaragua':'Americas',
    'Panama':'Americas', 'Paraguay':'Americas',
    'Peru':'Americas', 'Saint Kitts and Nevis':'Americas',
    'Saint Lucia':'Americas', 'Saint Vincent and the Grenadines':'Americas',
    'Suriname':'Americas', 'Trinidad and Tobago':'Americas',
    'United States':'Americas', 'Uruguay':'Americas',
    'Venezuela, RB':'Americas',

    # Oceania
    'Australia':'Oceania', 'Cook Islands':'Oceania',
    'Fiji':'Oceania', 'Kiribati':'Oceania',
    'Marshall Islands':'Oceania', 'Micronesia, Fed. Sts.':'Oceania',
    'Nauru':'Oceania', 'New Zealand':'Oceania',
    'Niue':'Oceania', 'Palau':'Oceania',
    'Papua New Guinea':'Oceania', 'Samoa':'Oceania',
    'Solomon Islands':'Oceania', 'Tonga':'Oceania',
    'Tuvalu':'Oceania', 'Vanuatu':'Oceania',
}

# ── Add region column to WHO dataset ──────────────────────────────
who['Region'] = who['Country'].map(region_map)

# ── Check for any unmapped countries ──────────────────────────────
unmapped = who[who['Region'].isnull()]['Country'].unique()
print(f"Unmapped countries: {len(unmapped)}")
print(unmapped)

In [ ]:
# ── Color map for regions ──────────────────────────────────────────
region_colors = {
    'Sub-Saharan Africa'      : '#e74c3c',
    'North Africa & Middle East' : '#e67e22',
    'Europe & Central Asia'   : '#3498db',
    'South & East Asia'       : '#2ecc71',
    'Americas'                : '#9b59b6',
    'Oceania'                 : '#1abc9c'
}

# ── Plot ───────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(12, 7))

for region, group in who.groupby('Region'):
    ax.scatter(
        group['GDP_per_capita'],
        group['Life expectancy'],
        label      = region,
        color      = region_colors[region],
        alpha      = 0.4,
        s          = 15
    )

ax.set_xscale('log')
ax.set_xlabel('GDP per Capita (USD) — Log Scale')
ax.set_ylabel('Life Expectancy (Years)')
ax.set_title('Life Expectancy vs GDP per Capita by Region\nWHO Dataset 2000–2015')
ax.legend(title='Region', bbox_to_anchor=(1.05, 1), loc='upper left')

plt.tight_layout()
plt.savefig(os.path.join(CHART_DIR_WHO, 'life_exp_vs_gdp_region.png'),
            dpi=150, bbox_inches='tight')
plt.show()

print("Chart saved.")

In [ ]:
import numpy as np

# ── Scatter with log-scale trend line ─────────────────────────────
fig, ax = plt.subplots(figsize=(12, 7))

for region, group in who.groupby('Region'):
    ax.scatter(
        group['GDP_per_capita'],
        group['Life expectancy'],
        label      = region,
        color      = region_colors[region],
        alpha      = 0.4,
        s          = 15
    )

# ── Fit trend line on log-transformed GDP ─────────────────────────
clean = who[['GDP_per_capita', 'Life expectancy']].dropna()
log_gdp = np.log10(clean['GDP_per_capita'])
z = np.polyfit(log_gdp, clean['Life expectancy'], 1)
p = np.poly1d(z)

x_line = np.linspace(log_gdp.min(), log_gdp.max(), 300)
ax.plot(10**x_line, p(x_line), color='black', linewidth=2, label='Trend line')

ax.set_xscale('log')
ax.set_xlabel('GDP per Capita (USD) — Log Scale')
ax.set_ylabel('Life Expectancy (Years)')
ax.set_title('Life Expectancy vs GDP per Capita — With Trend Line\nWHO Dataset 2000–2015')
ax.legend(title='Region', bbox_to_anchor=(1.05, 1), loc='upper left')

plt.tight_layout()
plt.savefig(os.path.join(CHART_DIR_WHO, 'life_exp_vs_gdp_trendline.png'),
            dpi=150, bbox_inches='tight')
plt.show()

print("Chart saved.")

In [21]:
# ── Identify high GDP Sub-Saharan African countries ────────────────
ssa = who[who['Region'] == 'Sub-Saharan Africa']

print("Sub-Saharan Africa GDP per capita > 5000:")
print(ssa[ssa['GDP_per_capita'] > 5000][['Country', 'Year', 'GDP_per_capita', 'Life expectancy']]
      .sort_values('GDP_per_capita', ascending=False)
      .drop_duplicates('Country')
      .to_string())

KeyError: 'Region'